<a href="https://colab.research.google.com/github/Tamanna0612/-Flyrank-internship-ML-Tamanna-/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tamanna0612/-Flyrank-internship-ML-Tamanna-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Ranking (via Scoring)

My lane is the Content Refresh Pipeline. While we are technically predicting a binary class (declining or not), the ultimate goal is Ranking. A content team has limited bandwidth, so they don't just need to know if a page is declining; they need a prioritized queue scoring the pages from highest risk to lowest risk so they know exactly what to review first.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd

# Setup Phase to prevent FileNotFoundError
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    if os.getcwd().split('/')[-1] != REPO_DIR:
        os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Dataset loaded. Total rows to rank: {df.shape[0]}")

Dataset loaded. Total rows to rank: 30000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: We want to predict if a page is losing its search footprint.

Proxy Label: Since "needs a refresh" is subjective, our hard label is a defined rule based on an observed outcome: trend_direction == 'down'. We convert this text column into a binary 1/0 label (1 for declining, 0 for stable/growing).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Creating the proxy label
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

decline_rate = df["is_declining_label"].mean()
print(f"Proxy label created. Overall baseline decline rate is: {decline_rate:.3f} (or {decline_rate*100:.1f}%)")

Proxy label created. Overall baseline decline rate is: 0.542 (or 54.2%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success Metric: Precision@K (e.g., Precision@50)

Accuracy is not a good metric here because a team will never review all 30,000 pages. If the content team only has time to update 50 pages this week, we only care about the density of true positives at the very top of our ranked list. A 'good' number is anything significantly higher than the baseline rate (or higher than a basic hand-written rule).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Showing why Precision@K matters:
# If we picked 50 pages randomly, we'd only get this many declining pages right:
random_precision = df["is_declining_label"].mean()
expected_right_out_of_50 = round(random_precision * 50)

print(f"Without a model (random guessing), Precision@50 would be ~{random_precision:.2f}")
print(f"That means only {expected_right_out_of_50} out of 50 reviewed pages would actually be declining.")
print("Our ML model needs to significantly beat this number.")

Without a model (random guessing), Precision@50 would be ~0.54
That means only 27 out of 50 reviewed pages would actually be declining.
Our ML model needs to significantly beat this number.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: One row = One specific web page (URL) at a snapshot in time, aggregating its historical performance (impressions, clicks) and content metadata (word count, age).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Displaying the unit of analysis (one row = one page)
print(f"Showing 1 row to confirm the unit of analysis ({len(df.columns)} columns of features per page):")
df.head(1)

Showing 1 row to confirm the unit of analysis (45 columns of features per page):


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-statement (like IF age > 365 days AND traffic < 100 THEN refresh) is too rigid. Some old pages are evergreen and never decline, while some new pages lose traffic rapidly due to seasonality or intense competition. An ML model (like a Decision Tree) can weigh complex, non-linear combinations of age, CTR, and position simultaneously—finding the exact multidimensional thresholds where a page starts to fail, which a human could never hardcode perfectly.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Let's prove why a simple 'if age > 365 days' rule fails by looking at the actual data

old_pages = df[df['content_age_days'] > 365]
new_pages = df[df['content_age_days'] <= 365]

old_decline_rate = old_pages['is_declining_label'].mean() * 100
new_decline_rate = new_pages['is_declining_label'].mean() * 100

print(f"Are all old pages declining? No. Only {old_decline_rate:.1f}% of pages older than a year are declining.")
print("This means a strict 'refresh old pages' rule would waste time on the other ~half which are evergreen.\n")

print(f"Are all new pages safe? No. {new_decline_rate:.1f}% of pages newer than a year are already declining.")
print("A strict rule would completely miss these failing new pages.\n")

print("Conclusion: The relationship is too messy for simple IF statements. We need ML to find the true signal.")

Are all old pages declining? No. Only 42.6% of pages older than a year are declining.
This means a strict 'refresh old pages' rule would waste time on the other ~half which are evergreen.

Are all new pages safe? No. 57.3% of pages newer than a year are already declining.
A strict rule would completely miss these failing new pages.

Conclusion: The relationship is too messy for simple IF statements. We need ML to find the true signal.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.